In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib as plt
import seaborn as sns

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
!pip install -q faiss-cpu

import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.1 MB/s eta 0:00:00:00:0100:01


In [5]:
train=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [6]:
import pandas as pd
import numpy as np
import string

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)
from sklearn.metrics.pairwise import cosine_similarity



In [7]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [8]:
print("Creating knowledge base...")

kb = []

for _, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

Creating knowledge base...


In [9]:
print("Loading embedding model and creating index...")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

kb_embeddings = embedding_model.encode(
    kb,
    show_progress_bar=False,
    convert_to_numpy=True
)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created!")
print(f"Knowledge Base Size : {len(kb)}")
print(f"Embedding Dimension : {kb_embeddings.shape[1]}")
print(f"FAISS Index Size    : {index.ntotal}")

Loading embedding model and creating index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base successfully created!
Knowledge Base Size : 2000
Embedding Dimension : 384
FAISS Index Size    : 2000


## Q1


In [10]:
from transformers import pipeline

# Load Zero-shot classifier (if not already loaded)
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Select row index 150
row_150 = train.iloc[150]

prompt_150 = str(row_150["prompt"])

labels_150 = [
    str(row_150["A"]),
    str(row_150["B"]),
    str(row_150["C"]),
    str(row_150["D"]),
    str(row_150["E"])
]

# Ground truth
correct_letter = row_150["answer"]
correct_option = str(row_150[correct_letter])

# Run zero-shot classification
result = zs(
    prompt_150,
    candidate_labels=labels_150,
    multi_label=False
)

# Display complete ranking
print("Prediction Ranking:")
for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.6f}  -->  {label}")

# Probability assigned to the correct option
correct_score = dict(zip(result["labels"], result["scores"]))[correct_option]

print("\nGround Truth Letter :", correct_letter)
print("Ground Truth Option :", correct_option)
print(f"Required Answer (3 decimals): {correct_score:.3f}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Prediction Ranking:
0.384420  -->  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.378677  -->  The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.092697  -->  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity."
0.078618  -->  The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Ess

## Q2

In [13]:


row_idx = 150

# Prompt to query
query = str(train.iloc[row_idx]["prompt"])

# Embed the prompt
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

# Search FAISS
k = 10
distances, indices = index.search(query_embedding, k)

print("Top-10 Retrieved KB Indices:")
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), start=1):
    print(f"Rank {rank}: KB Index = {idx}, Distance = {dist:.4f}")

# Determine the rank of the true document
true_index = row_idx

if true_index in indices[0]:
    rank = np.where(indices[0] == true_index)[0][0] + 1
    print(f"\n Required Answer: {rank}")
else:
    print("\n The true document is NOT in the Top-10 retrieved results.")

Top-10 Retrieved KB Indices:
Rank 1: KB Index = 663, Distance = 0.2639
Rank 2: KB Index = 1701, Distance = 0.2639
Rank 3: KB Index = 1269, Distance = 0.2664
Rank 4: KB Index = 1532, Distance = 0.2664
Rank 5: KB Index = 576, Distance = 0.2686
Rank 6: KB Index = 847, Distance = 0.2699
Rank 7: KB Index = 1693, Distance = 0.2699
Rank 8: KB Index = 1906, Distance = 0.2699
Rank 9: KB Index = 168, Distance = 0.2832
Rank 10: KB Index = 150, Distance = 0.2871

 Answer: 10


## Q3

In [14]:
from sentence_transformers import CrossEncoder
import numpy as np



row_idx = 150

# Prompt
prompt_150 = str(train.iloc[row_idx]["prompt"])

# Reuse the FAISS retrieval from Q2
query_embedding = embedding_model.encode(
    [prompt_150],
    convert_to_numpy=True
)

k = 10
distances, indices = index.search(query_embedding, k)

retrieved_indices = indices[0]

# Retrieve the corresponding documents
docs_10 = [kb[i] for i in retrieved_indices]

# Load Cross-Encoder
cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

# Create (query, document) pairs
pairs = [[prompt_150, doc] for doc in docs_10]

# Predict relevance scores
ce_scores = cross_encoder.predict(pairs)

# Sort by score (highest first)
ranking = sorted(
    zip(retrieved_indices, docs_10, ce_scores),
    key=lambda x: x[2],
    reverse=True
)

print("Cross-Encoder Re-ranked Results:\n")

for rank, (idx, doc, score) in enumerate(ranking, start=1):
    print(f"Rank {rank}: KB Index = {idx}, Score = {score:.4f}")

# Find the rank of the true document
true_index = row_idx

true_rank = None
for rank, (idx, _, _) in enumerate(ranking, start=1):
    if idx == true_index:
        true_rank = rank
        break

if true_rank is not None:
    print(f"\n✅ Required Answer: {true_rank}")
else:
    print("\n❌ The true document is not present in the retrieved Top-10.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-Encoder Re-ranked Results:

Rank 1: KB Index = 150, Score = 4.7585
Rank 2: KB Index = 847, Score = 4.7526
Rank 3: KB Index = 1693, Score = 4.7526
Rank 4: KB Index = 1906, Score = 4.7526
Rank 5: KB Index = 1269, Score = 4.7375
Rank 6: KB Index = 1532, Score = 4.7375
Rank 7: KB Index = 168, Score = 4.7072
Rank 8: KB Index = 576, Score = 4.6870
Rank 9: KB Index = 663, Score = 4.6602
Rank 10: KB Index = 1701, Score = 4.6602

✅ Required Answer: 1


## Q4

In [16]:
from transformers import AutoTokenizer



row_idx = 42
k = 5

# Prompt
prompt = str(train.iloc[row_idx]["prompt"])

# Embed the prompt
query_embedding = embedding_model.encode(
    [prompt],
    convert_to_numpy=True
)

# Retrieve Top-5 documents from FAISS
distances, indices = index.search(query_embedding, k)

retrieved_indices = indices[0]
retrieved_docs = [kb[i] for i in retrieved_indices]

# Concatenate documents with a single space
context = " ".join(retrieved_docs)

# Create the required input string
input_text = f"Context: {context} Question: {prompt}"

# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenize WITHOUT truncation
tokens = tokenizer(
    input_text,
    truncation=False,
    add_special_tokens=True
)

# Count total tokens
num_tokens = len(tokens["input_ids"])

print("Retrieved KB Indices:", retrieved_indices.tolist())
print("Total Tokens:", num_tokens)

Retrieved KB Indices: [241, 439, 456, 506, 605]
Total Tokens: 216


## Q5

In [17]:


row_idx = 150

# Original prompt
prompt_150 = str(train.iloc[row_idx]["prompt"])

# Retrieve the TRUE document from the KB
true_document = kb[row_idx]

# Create the RAG input
rag_input = f"Context: {true_document} Question: {prompt_150}"

# Candidate labels (A-E)
labels_150 = [
    str(train.iloc[row_idx]["A"]),
    str(train.iloc[row_idx]["B"]),
    str(train.iloc[row_idx]["C"]),
    str(train.iloc[row_idx]["D"]),
    str(train.iloc[row_idx]["E"])
]

# Ground-truth
correct_letter = train.iloc[row_idx]["answer"]
correct_option = str(train.iloc[row_idx][correct_letter])

# Run zero-shot classification
result = zs(
    rag_input,
    candidate_labels=labels_150,
    multi_label=False
)

# Display ranking
print("Prediction Ranking:\n")
for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.6f} --> {label}")

# Score assigned to the correct option
correct_score = dict(zip(result["labels"], result["scores"]))[correct_option]

print("\nGround Truth Letter :", correct_letter)
print("Ground Truth Option :", correct_option)
print(f"\n Required Answer: {correct_score:.3f}")

Prediction Ranking:

0.989426 --> The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.004490 --> The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.002796 --> The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.001709 --> The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Rela

## Q6

In [18]:


row_idx = 150

# Original prompt
prompt_150 = str(train.iloc[row_idx]["prompt"])

# Deliberately use an unrelated document
wrong_document = kb[999]

# Create the adversarial RAG input
adversarial_input = f"Context: {wrong_document} Question: {prompt_150}"

# Candidate labels
labels_150 = [
    str(train.iloc[row_idx]["A"]),
    str(train.iloc[row_idx]["B"]),
    str(train.iloc[row_idx]["C"]),
    str(train.iloc[row_idx]["D"]),
    str(train.iloc[row_idx]["E"])
]

# Ground truth
correct_letter = train.iloc[row_idx]["answer"]
correct_option = str(train.iloc[row_idx][correct_letter])

# Run zero-shot classifier
result = zs(
    adversarial_input,
    candidate_labels=labels_150,
    multi_label=False
)

# Display prediction ranking
print("Prediction Ranking:\n")
for label, score in zip(result["labels"], result["scores"]):
    print(f"{score:.6f} --> {label}")

# Probability assigned to the correct option
score_dict = dict(zip(result["labels"], result["scores"]))
correct_score = score_dict[correct_option]

print("\nGround Truth Letter :", correct_letter)
print("Ground Truth Option :", correct_option)
print("\nInjected Context (KB Index 999):")
print(wrong_document)

print(f"\n Required Answer: {correct_score:.3f}")

Prediction Ranking:

0.528949 --> The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.425296 --> The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.020378 --> The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.018688 --> The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical framework has no effect on subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
0.006689 --> The butterfly effect is the phenomenon that a small change i

## Q7

In [19]:

k = 5
hits = 0
total = 100

for row_idx in range(total):

    # Prompt
    prompt = str(train.iloc[row_idx]["prompt"])

    # Ground-truth correct document (exact answer text)
    correct_letter = train.iloc[row_idx]["answer"]
    correct_doc = str(train.iloc[row_idx][correct_letter])

    # Embed prompt
    query_embedding = embedding_model.encode(
        [prompt],
        convert_to_numpy=True
    )

    # Retrieve Top-k documents
    distances, indices = index.search(query_embedding, k)

    retrieved_docs = [kb[i] for i in indices[0]]

    # Count as hit if exact document is retrieved
    if correct_doc in retrieved_docs:
        hits += 1

hit_rate = (hits / total) * 100

print(f"Hits: {hits}/{total}")
print(f"Hit Rate@{k}: {hit_rate:.1f}%")

Hits: 73/100
Hit Rate@5: 73.0%


## Q8

In [20]:


import numpy as np

# MAP@3 function
def map3(actual, predictions):
    """
    actual: correct answer letter (A-E)
    predictions: list of top-3 predicted letters
    """
    for rank, pred in enumerate(predictions[:3], start=1):
        if pred == actual:
            return 1.0 / rank
    return 0.0


cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

scores = []

for row_idx in range(20):

    # Retrieve
    row = train.iloc[row_idx]

    prompt = str(row["prompt"])

    query_embedding = embedding_model.encode(
        [prompt],
        convert_to_numpy=True
    )

    distances, indices = index.search(query_embedding, 5)

    retrieved_indices = indices[0]
    retrieved_docs = [kb[i] for i in retrieved_indices]

    # Rerank
    pairs = [[prompt, doc] for doc in retrieved_docs]

    ce_scores = cross_encoder.predict(pairs)

    best_doc = retrieved_docs[np.argmax(ce_scores)]

    # Augment
    rag_prompt = f"Context: {best_doc} Question: {prompt}"

    # Predict
    option_texts = {
        "A": str(row["A"]),
        "B": str(row["B"]),
        "C": str(row["C"]),
        "D": str(row["D"]),
        "E": str(row["E"])
    }

    candidate_labels = list(option_texts.values())

    result = zs(
        rag_prompt,
        candidate_labels=candidate_labels,
        multi_label=False
    )

    # Convert option text -> option letter
    text_to_letter = {v: k for k, v in option_texts.items()}

    ranked_letters = [
        text_to_letter[label]
        for label in result["labels"]
    ]

    top3 = ranked_letters[:3]

    # MAP@3
    score = map3(
        row["answer"],
        top3
    )

    scores.append(score)

    print(
        f"Row {row_idx:2d} | "
        f"GT={row['answer']} | "
        f"Pred={top3} | "
        f"MAP@3={score:.3f}"
    )

# Final Score
final_map3 = np.mean(scores)

print(f"Average MAP@3: {final_map3:.3f}")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Row  0 | GT=B | Pred=['B', 'D', 'A'] | MAP@3=1.000
Row  1 | GT=A | Pred=['A', 'E', 'C'] | MAP@3=1.000
Row  2 | GT=C | Pred=['C', 'D', 'B'] | MAP@3=1.000
Row  3 | GT=B | Pred=['B', 'D', 'A'] | MAP@3=1.000
Row  4 | GT=A | Pred=['A', 'B', 'C'] | MAP@3=1.000
Row  5 | GT=C | Pred=['B', 'C', 'A'] | MAP@3=0.500
Row  6 | GT=E | Pred=['E', 'B', 'D'] | MAP@3=1.000
Row  7 | GT=A | Pred=['A', 'B', 'C'] | MAP@3=1.000
Row  8 | GT=A | Pred=['A', 'C', 'D'] | MAP@3=1.000
Row  9 | GT=A | Pred=['A', 'B', 'C'] | MAP@3=1.000
Row 10 | GT=C | Pred=['C', 'A', 'D'] | MAP@3=1.000
Row 11 | GT=B | Pred=['B', 'C', 'A'] | MAP@3=1.000
Row 12 | GT=D | Pred=['D', 'A', 'B'] | MAP@3=1.000
Row 13 | GT=E | Pred=['E', 'D', 'B'] | MAP@3=1.000
Row 14 | GT=E | Pred=['E', 'A', 'D'] | MAP@3=1.000
Row 15 | GT=E | Pred=['E', 'B', 'C'] | MAP@3=1.000
Row 16 | GT=C | Pred=['C', 'B', 'E'] | MAP@3=1.000
Row 17 | GT=C | Pred=['C', 'B', 'E'] | MAP@3=1.000
Row 18 | GT=B | Pred=['B', 'D', 'A'] | MAP@3=1.000
Row 19 | GT=C | Pred=['C', 'D',